<a href="https://colab.research.google.com/github/kushal984762333-glitch/ibm-ds-capstone-spacex/blob/main/07-predictive-analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 07 - Predictive Analysis (Classification)
Logistic Regression, SVM, Decision Tree and KNN tuned with GridSearchCV.

In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt, seaborn as sns
from sklearn import preprocessing
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import confusion_matrix
data=pd.read_csv('https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-DS0701EN-SkillsNetwork/api/dataset_part_2.csv')
X=pd.read_csv('https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-DS0701EN-SkillsNetwork/api/dataset_part_3.csv')
Y=data['Class'].to_numpy()
X=preprocessing.StandardScaler().fit(X).transform(X)
X_train,X_test,Y_train,Y_test=train_test_split(X,Y,test_size=0.2,random_state=2)
print(X_test.shape)

In [ ]:
def plot_cm(y,yhat,title):
    cm=confusion_matrix(y,yhat);ax=plt.subplot();sns.heatmap(cm,annot=True,ax=ax,fmt='d')
    ax.set_xlabel('Predicted');ax.set_ylabel('True');ax.set_title(title)
    ax.xaxis.set_ticklabels(['did not land','landed']);ax.yaxis.set_ticklabels(['did not land','landed']);plt.show()
models={
 'Logistic Regression':(LogisticRegression(),{'C':[0.01,0.1,1],'penalty':['l2'],'solver':['lbfgs']}),
 'SVM':(SVC(),{'kernel':['linear','rbf','poly','sigmoid'],'C':np.logspace(-3,3,5),'gamma':np.logspace(-3,3,5)}),
 'Decision Tree':(DecisionTreeClassifier(),{'criterion':['gini','entropy'],'splitter':['best','random'],'max_depth':[2*n for n in range(1,10)],'max_features':['sqrt','log2'],'min_samples_leaf':[1,2,4],'min_samples_split':[2,5,10]}),
 'KNN':(KNeighborsClassifier(),{'n_neighbors':list(range(1,11)),'algorithm':['auto','ball_tree','kd_tree','brute'],'p':[1,2]})}
results={}
for name,(est,grid) in models.items():
    gs=GridSearchCV(est,grid,cv=10).fit(X_train,Y_train)
    results[name]=(gs.best_score_,gs.score(X_test,Y_test))
    print(name,gs.best_params_,'CV acc:',round(gs.best_score_,4),'Test acc:',round(gs.score(X_test,Y_test),4))
    plot_cm(Y_test,gs.predict(X_test),name)

In [ ]:
res=pd.DataFrame(results,index=['CV accuracy','Test accuracy']).T
res.plot(kind='bar',ylim=(0.5,1));plt.title('Model comparison');plt.show()
print('Best model by CV:',res['CV accuracy'].idxmax())

In [2]:
!pip install dash plotly pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 31.4 MB/s eta 0:00:00


In [3]:
import pandas as pd
import plotly.express as px
from dash import Dash, dcc, html
from dash.dependencies import Input, Output

URL = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/spacex_launch_dash.csv"
spacex_df = pd.read_csv(URL)
min_p, max_p = spacex_df["Payload Mass (kg)"].min(), spacex_df["Payload Mass (kg)"].max()

app = Dash(__name__)
sites = [{"label": "All Sites", "value": "ALL"}] + [
    {"label": s, "value": s} for s in spacex_df["Launch Site"].unique()]

app.layout = html.Div([
    html.H1("SpaceX Launch Records Dashboard", style={"textAlign": "center"}),
    dcc.Dropdown(id="site-dropdown", options=sites, value="ALL",
                 placeholder="Select a Launch Site", searchable=True),
    html.Br(),
    dcc.Graph(id="success-pie-chart"),
    html.P("Payload range (kg):"),
    dcc.RangeSlider(id="payload-slider", min=0, max=10000, step=1000,
                    marks={i: str(i) for i in range(0, 10001, 2500)}, value=[min_p, max_p]),
    dcc.Graph(id="success-payload-scatter-chart"),
])

@app.callback(Output("success-pie-chart", "figure"), Input("site-dropdown", "value"))
def pie(site):
    if site == "ALL":
        return px.pie(spacex_df, values="class", names="Launch Site", title="Total Successful Launches by Site")
    d = spacex_df[spacex_df["Launch Site"] == site]["class"].value_counts().reset_index()
    d.columns = ["class", "count"]
    return px.pie(d, values="count", names="class", title=f"Success vs Failure for {site}")

@app.callback(Output("success-payload-scatter-chart", "figure"),
              [Input("site-dropdown", "value"), Input("payload-slider", "value")])
def scatter(site, rng):
    d = spacex_df[spacex_df["Payload Mass (kg)"].between(rng[0], rng[1])]
    if site != "ALL":
        d = d[d["Launch Site"] == site]
    return px.scatter(d, x="Payload Mass (kg)", y="class", color="Booster Version Category",
                      title="Payload vs Launch Outcome")

if __name__ == "__main__":
    app.run(debug=True)

<IPython.core.display.Javascript object>